# **Fine-Tuning a Casual Language Model with LoRA**
---
This notebook demonstrates how to fine-tune a Causal Language Model (CLM) like bigscience/bloom-560m using the LoRA technique, which significantly reduces the computational and memory requirements for fine-tuning large models.

**Note**: For the demo purpose, I use the open source mode "bloom-560m". The ideal model should be a bigger model, like gemini-2.5-flash, but need a Google Cloud Platform (GCP) account and project (Vertex AI) and billing enabled.

**Setup and Installation** 🛠️

Install necessary Python packages.

In [ ]:
# Install required libraries
!pip install transformers accelerate datasets peft bitsandbytes safetensors

**Google Drive Mounting** 💾

Mount your Google Drive to access your dataset file and save the fine-tuned model weights.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

**Import Libraries** 📚

Import all necessary modules from the installed libraries.

In [ ]:
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
    default_data_collator
)
from peft import LoraConfig, get_peft_model

**Configuration Variables** ⚙️

Set file paths, model names, and training hyperparameters

In [ ]:
# Path to your JSONL dataset file on Google Drive
DATASET_PATH = "/content/drive/training/autism_knowledge_base.jsonl"

# Base model to be fine-tuned (e.g., BLOOM-560M)
MODEL_NAME = "bigscience/bloom-560m"

# Directory where training checkpoints will be saved locally
OUTPUT_DIR = "./tunedModels"

# Name for the final fine-tuned model weights (used for saving and zipping)
TUNNING_MODEL_NAME = "autism-knowledge-base-v1"

MAX_LENGTH = 512    # Maximum sequence length for tokenization
BATCH_SIZE = 4      # Effective batch size (before gradient accumulation)
EPOCHS = 3          # Number of training epochs

**Load and Preprocess Dataset** 📄

Load the dataset from the specified path and map the columns to the expected format.

In [ ]:
# Load the dataset from the JSONL file
dataset = load_dataset("json", data_files=DATASET_PATH)

# Function to rename columns from the loaded JSONL to a standard format
def preprocess(example):
    return {"input_text": example["text_input"], "target_text": example["output"]}

# Apply the preprocessing function to the 'train' split
dataset = dataset["train"].map(preprocess)

print(dataset)

**Initialize Tokenizer and Data Collator** 📝

Load the tokenizer for the base model and configure it

In [ ]:
# Load the tokenizer from the pretrained model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set the padding token to the EOS token
tokenizer.pad_token = tokenizer.eos_token

# Initialize the data collator for language modeling (MLM=False for Causal LM)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

**Load Model with 8-bit Quantization and Apply LoRA** 🚀

Load the model using 8-bit quantization (BitsAndBytes) to save GPU memory. Then, apply the LoRA configuration to enable parameter-efficient fine-tuning.

In [ ]:
# 1. Setup 8-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# 2. Load the base model with the quantization config
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config
)

# 3. Define the LoRA configuration
lora_config = LoraConfig(
    r=8,                       # LoRA attention dimension
    lora_alpha=32,             # Alpha parameter for LoRA scaling
    target_modules=["query_key_value"], # Modules to apply LoRA to (specific to BLOOM)
    lora_dropout=0.1,          # Dropout probability for LoRA layers
    bias="none",               # Bias type
    task_type="CAUSAL_LM"      # Task type
)

# 4. Wrap the model with the LoRA configuration
model = get_peft_model(model, lora_config)

# Print trainable parameters to confirm LoRA is active
model.print_trainable_parameters()

**Tokenize the Dataset for Training** ✂️

Define the tokenization function.

In [ ]:
# Define the maximum sequence length
max_length = MAX_LENGTH

def tokenize_fn(example):
    # Tokenize the input text (Prompt)
    model_inputs = tokenizer(
        example["input_text"],
        max_length=max_length,
        padding="max_length",
        truncation=True
    )

    # Tokenize the target text (Expected Output) to be used as labels
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            example["target_text"],
            max_length=max_length,
            padding="max_length",
            truncation=True
        )

    # Set the 'labels' key in the model inputs
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the tokenization function to the entire dataset
tokenized_dataset = dataset.map(tokenize_fn, batched=True)

**Configure Training Arguments** 📊

Set up the TrainingArguments, including output directory, batch size, learning rate, and saving strategy.

In [ ]:
# Configure the training run
training_args = TrainingArguments(
    output_dir="./bloom-lora",               # Output directory for checkpoints
    per_device_train_batch_size=2,           # Batch size per device
    gradient_accumulation_steps=8,           # Accumulate gradients over 8 steps
    num_train_epochs=EPOCHS,                 # Total number of training epochs
    learning_rate=2e-4,                      # Learning rate
    fp16=True,                               # Use 16-bit precision for speed
    logging_steps=10,                        # Log training metrics every 10 steps
    save_total_limit=2,                      # Only keep the last 2 checkpoints
    save_strategy="epoch",                   # Save checkpoints at the end of each epoch
    report_to=[]                             # Disable logging to services like W&B
)

**Initialize and Run Trainer** 🏋️

Create the Trainer object and start the fine-tuning process.

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=default_data_collator, # Use the default data collator for this tokenized format
)

# Start training!
print("Starting training...")
trainer.train()
print("Training complete.")

**Save and Zip the Fine-Tuned Model Weights** 📦

After training, save the final LoRA adapter weights and the tokenizer. Then, compress them into a single ZIP file for easy download or transfer.

In [ ]:
# Save the PEFT (LoRA) weights and the tokenizer
model.save_pretrained(TUNNING_MODEL_NAME)
tokenizer.save_pretrained(TUNNING_MODEL_NAME)

print(f"Model and tokenizer saved to directory: {TUNNING_MODEL_NAME}")

In [ ]:
# Zip the saved model directory for easy download

!zip -r {TUNNING_MODEL_NAME}.zip {TUNNING_MODEL_NAME}

print(f"Zipped file created: {TUNNING_MODEL_NAME}.zip")